In [ ]:
# Pixelforge project configuration
from pathlib import Path

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Open Jupyter from the Pixelforge project or notebooks directory.")

PROJECT_ROOT = find_project_root()
PROJECT_DIR = PROJECT_ROOT

IMAGE_DIR = PROJECT_ROOT / "dataset" / "images"
VIDEO_DIR = PROJECT_ROOT / "dataset" / "videos"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

print("Pixelforge root:", PROJECT_ROOT)
print("Images directory:", IMAGE_DIR)
print("Videos directory:", VIDEO_DIR)


In [ ]:
%pip install numpy matplotlib opencv-contrib-python scikit-image pandas


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path
from skimage.metrics import peak_signal_noise_ratio
from skimage.metrics import structural_similarity

print("All libraries imported successfully!")
print("OpenCV Version:", cv2.__version__)
print("NumPy Version:", np.__version__)


In [ ]:
# Main project directory
PROJECT_DIR = PROJECT_ROOT

# Dataset directory
DATASET_DIR = IMAGE_DIR

# Output directories
OUTPUT_DIR = PROJECT_DIR / "outputs"
UNIT3_DIR = OUTPUT_DIR / "unit_3"
UNIT4_DIR = OUTPUT_DIR / "unit_4"
UNIT5_DIR = OUTPUT_DIR / "unit_5_object_detection"

# Create directories
for folder in [DATASET_DIR, UNIT3_DIR, UNIT4_DIR, UNIT5_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directories created successfully!")

In [ ]:

print("Current Working Directory:")
print(os.getcwd())

print("\nFiles in Current Directory:")
print(os.listdir())import os

print("Current Working Directory:")
print(os.getcwd())

print("\nFiles in Current Directory:")
print(os.listdir())

In [ ]:
from pathlib import Path

DATASET_DIR = IMAGE_DIR

print("Dataset Path:", DATASET_DIR)
print("Dataset Exists:", DATASET_DIR.exists())

if DATASET_DIR.exists():
    print("\nFiles inside dataset:")
    for file in DATASET_DIR.iterdir():
        print(file.name)
    

In [ ]:
image_path_1 = DATASET_DIR / "original_image.jpg.jpg"
image_path_2 = DATASET_DIR / "second_image.jpg.jpg"

img1 = cv2.imread(str(image_path_1))
img2 = cv2.imread(str(image_path_2))

assert img1 is not None, f"Cannot load: {image_path_1}"
assert img2 is not None, f"Cannot load: {image_path_2}"

original = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
second = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

print("Both photographs loaded successfully!")


In [ ]:
from pathlib import Path

DATASET_DIR = IMAGE_DIR

print("Dataset folder exists:", DATASET_DIR.exists())
print("\nActual filenames:")

for file in DATASET_DIR.iterdir():
    print(
        "Name:", repr(file.name),
        "| Extension:", repr(file.suffix),
        "| Exists:", file.exists()
    )
    

In [ ]:
import cv2
from pathlib import Path

DATASET_DIR = IMAGE_DIR

# Find all image files
image_files = sorted([
    file for file in DATASET_DIR.iterdir()
    if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

print("Images detected:", len(image_files))

for file in image_files:
    print(file.name)

# Validate number of images
assert len(image_files) >= 2, "Please ensure both images are present."

# Load photographs
img1 = cv2.imread(str(image_files[0]))
img2 = cv2.imread(str(image_files[1]))

assert img1 is not None, "First image could not be loaded."
assert img2 is not None, "Second image could not be loaded."

# Convert BGR to RGB
original = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
second = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

print("\nBoth photographs loaded successfully!")


In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original Photograph")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(second)
plt.title("Second Photograph")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# UNIT 3: IMAGE RESTORATION AND SEGMENTATION

## Experiment 1: Image Restoration

### Objective
To study image restoration techniques by introducing Gaussian
noise and Salt-and-Pepper noise into an original image.

Median filtering and Gaussian filtering are applied to restore
the noisy images. PSNR and SSIM are calculated to evaluate
the quality of the restored images.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Convert original image to uint8
clean_image = original.astype(np.uint8)

# 1. Gaussian Noise
gaussian_noise = np.random.normal(
    loc=0,
    scale=25,
    size=clean_image.shape
)

gaussian_noisy = np.clip(
    clean_image.astype(np.float32) + gaussian_noise,
    0, 255
).astype(np.uint8)


# 2. Salt-and-Pepper Noise
salt_pepper = clean_image.copy()

probability = 0.04

random_matrix = np.random.random(clean_image.shape[:2])

salt_pepper[random_matrix < probability / 2] = [255, 255, 255]

salt_pepper[random_matrix > 1 - probability / 2] = [0, 0, 0]


print("Gaussian noise generated successfully.")
print("Salt-and-Pepper noise generated successfully.")


In [ ]:
# Gaussian noisy image restoration
gaussian_restored_median = cv2.medianBlur(
    gaussian_noisy, 5
)

gaussian_restored_blur = cv2.GaussianBlur(
    gaussian_noisy, (5, 5), 0
)


# Salt-and-Pepper noisy image restoration
sp_restored_median = cv2.medianBlur(
    salt_pepper, 5
)

sp_restored_blur = cv2.GaussianBlur(
    salt_pepper, (5, 5), 0
)


print("Median filtering completed.")
print("Gaussian filtering completed.")

In [ ]:
images = [
    (clean_image, "Original Image"),
    (gaussian_noisy, "Gaussian Noise"),
    (gaussian_restored_median, "Gaussian Noise - Median Filter"),
    (gaussian_restored_blur, "Gaussian Noise - Gaussian Filter"),
    (salt_pepper, "Salt & Pepper Noise"),
    (sp_restored_median, "Salt & Pepper - Median Filter"),
    (sp_restored_blur, "Salt & Pepper - Gaussian Filter")
]

plt.figure(figsize=(18, 12))

for i, (image, title) in enumerate(images):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def calculate_metrics(reference, processed):

    psnr = peak_signal_noise_ratio(
        reference, processed, data_range=255
    )

    ssim = structural_similarity(
        reference,
        processed,
        channel_axis=2,
        data_range=255
    )

    return round(psnr, 2), round(ssim, 4)


results = []

experiments = {
    "Gaussian Noise": gaussian_noisy,
    "Gaussian + Median": gaussian_restored_median,
    "Gaussian + Gaussian Filter": gaussian_restored_blur,
    "Salt & Pepper Noise": salt_pepper,
    "S&P + Median": sp_restored_median,
    "S&P + Gaussian Filter": sp_restored_blur
}

for name, image in experiments.items():

    psnr, ssim = calculate_metrics(
        clean_image, image
    )

    results.append({
        "Technique": name,
        "PSNR (dB)": psnr,
        "SSIM": ssim
    })

metrics_df = pd.DataFrame(results)

display(metrics_df)

In [ ]:
restoration_outputs = {
    "original.png": clean_image,
    "gaussian_noise.png": gaussian_noisy,
    "gaussian_median.png": gaussian_restored_median,
    "gaussian_filter.png": gaussian_restored_blur,
    "salt_pepper_noise.png": salt_pepper,
    "sp_median.png": sp_restored_median,
    "sp_gaussian.png": sp_restored_blur
}

for filename, image in restoration_outputs.items():

    cv2.imwrite(
        str(UNIT3_DIR / filename),
        cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    )

metrics_df.to_csv(
    UNIT3_DIR / "restoration_metrics.csv",
    index=False
)

print("All restoration outputs saved successfully.")

In [ ]:
### Observations

1. Gaussian noise introduces random intensity variations
   throughout the image.

2. Salt-and-Pepper noise introduces random black and white pixels.

3. Median filtering is generally effective for removing
   Salt-and-Pepper noise.

4. Gaussian filtering smooths random variations caused by
   Gaussian noise, but may blur image edges.

5. PSNR and SSIM are used to compare restored images with
   the original image.

### Conclusion

Image restoration techniques help reduce noise and improve
image quality. Median filtering is particularly suitable for
Salt-and-Pepper noise, while Gaussian filtering is useful
for smoothing Gaussian noise.

In [ ]:
## Experiment 2: Morphological Processing and Segmentation

### Objective

To perform image segmentation using thresholding and
morphological operations.

The experiment includes:
1. Binary Thresholding
2. Erosion
3. Dilation
4. Opening
5. Closing
6. Canny Edge Detection

These techniques help identify object boundaries,
remove noise and improve image segmentation.

In [ ]:
# Convert original image into grayscale
gray = cv2.cvtColor(clean_image, cv2.COLOR_RGB2GRAY)

# Apply Gaussian blur to reduce noise
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# Apply Otsu's thresholding
threshold_value, binary = cv2.threshold(
    blurred,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print("Threshold value:", threshold_value)
print("Binary thresholding completed.")

In [ ]:
# Create a structuring element (kernel)
kernel = np.ones((5, 5), np.uint8)

# 1. Erosion
erosion = cv2.erode(binary, kernel, iterations=1)

# 2. Dilation
dilation = cv2.dilate(binary, kernel, iterations=1)

# 3. Opening
opening = cv2.morphologyEx(
    binary, cv2.MORPH_OPEN, kernel
)

# 4. Closing
closing = cv2.morphologyEx(
    binary, cv2.MORPH_CLOSE, kernel
)

print("Erosion completed.")
print("Dilation completed.")
print("Opening completed.")
print("Closing completed.")

In [ ]:
edges = cv2.Canny(
    blurred,
    threshold1=100,
    threshold2=200
)

print("Canny edge detection completed.")

In [ ]:
results_morphology = [
    (gray, "Grayscale Image"),
    (binary, "Binary Thresholding"),
    (erosion, "Erosion"),
    (dilation, "Dilation"),
    (opening, "Opening"),
    (closing, "Closing"),
    (edges, "Canny Edge Detection")
]

plt.figure(figsize=(16, 12))

for i, (image, title) in enumerate(results_morphology):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image, cmap="gray")
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
morphology_outputs = {
    "grayscale.png": gray,
    "binary_threshold.png": binary,
    "erosion.png": erosion,
    "dilation.png": dilation,
    "opening.png": opening,
    "closing.png": closing,
    "canny_edges.png": edges
}

for filename, image in morphology_outputs.items():
    cv2.imwrite(str(UNIT3_DIR / filename), image)

print("Morphological processing outputs saved.")

In [ ]:
## Experiment 3: Watershed Segmentation

### Objective

To implement watershed segmentation for identifying
foreground objects and separating regions using
distance transformation and marker-based segmentation.

The algorithm identifies foreground markers, background
regions and unknown regions before applying watershed.

In [ ]:
# Convert image to grayscale
watershed_gray = cv2.cvtColor(
    clean_image, cv2.COLOR_RGB2GRAY
)

# Remove noise
watershed_blur = cv2.GaussianBlur(
    watershed_gray, (5, 5), 0
)

# Binary thresholding
_, watershed_binary = cv2.threshold(
    watershed_blur,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

# Remove small noise
kernel = np.ones((3, 3), np.uint8)

opening_ws = cv2.morphologyEx(
    watershed_binary,
    cv2.MORPH_OPEN,
    kernel,
    iterations=2
)

# Identify sure background
sure_bg = cv2.dilate(
    opening_ws, kernel, iterations=3
)

# Distance transformation
dist_transform = cv2.distanceTransform(
    opening_ws, cv2.DIST_L2, 5
)

# Identify sure foreground
_, sure_fg = cv2.threshold(
    dist_transform,
    0.4 * dist_transform.max(),
    255,
    0
)

sure_fg = np.uint8(sure_fg)

# Identify unknown regions
unknown = cv2.subtract(
    sure_bg, sure_fg
)

print("Foreground and background regions identified.")

In [ ]:
# Label foreground components
num_labels, markers = cv2.connectedComponents(
    sure_fg
)

# Shift labels so background is not zero
markers = markers + 1

# Mark unknown regions as zero
markers[unknown == 255] = 0

# Apply watershed
watershed_input = cv2.cvtColor(
    clean_image, cv2.COLOR_RGB2BGR
)

markers = cv2.watershed(
    watershed_input, markers
)

# Highlight watershed boundaries in red
watershed_result = clean_image.copy()

watershed_result[markers == -1] = [255, 0, 0]

print("Watershed segmentation completed.")
print("Number of marker labels:", num_labels)

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(clean_image)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(sure_fg, cmap="gray")
plt.title("Sure Foreground")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(watershed_result)
plt.title("Watershed Segmentation")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
cv2.imwrite(
    str(UNIT3_DIR / "watershed_foreground.png"),
    sure_fg
)

cv2.imwrite(
    str(UNIT3_DIR / "watershed_result.png"),
    cv2.cvtColor(
        watershed_result, cv2.COLOR_RGB2BGR
    )
)

print("Watershed outputs saved successfully.")

In [ ]:
## Unit 3: Conclusion

Image restoration and segmentation techniques were
successfully implemented using OpenCV.

Gaussian and Salt-and-Pepper noise were introduced,
and median and Gaussian filters were used for restoration.

PSNR and SSIM measurements were calculated to compare
restoration quality.

Morphological operations including erosion, dilation,
opening and closing were performed.

Canny edge detection and Watershed segmentation were
also implemented.

The experiments demonstrate how image processing
techniques can improve image quality and identify
important regions and boundaries.

In [ ]:
# UNIT 4: FEATURE EXTRACTION

## Experiment 4: HOG Feature Extraction

### Objective

To extract Histogram of Oriented Gradients (HOG) features
from the input photograph.

HOG is a feature descriptor that represents an image using
the distribution of gradient orientations. It is commonly
used in object detection and image recognition.

In [ ]:
from skimage.feature import hog
from skimage import exposure

# Convert image to grayscale
hog_gray = cv2.cvtColor(
    clean_image, cv2.COLOR_RGB2GRAY
)

# Extract HOG features
hog_features, hog_image = hog(
    hog_gray,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True,
    block_norm="L2-Hys"
)

# Improve visualization contrast
hog_visual = exposure.rescale_intensity(
    hog_image,
    in_range=(0, 10)
)

print("HOG feature extraction completed!")
print("HOG descriptor shape:", hog_features.shape)
print("Total HOG features:", len(hog_features))

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(clean_image)
plt.title("Original Photograph")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(hog_visual, cmap="gray")
plt.title("HOG Feature Visualization")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.imsave(
    UNIT4_DIR / "hog_visualization.png",
    hog_visual,
    cmap="gray"
)

np.save(
    UNIT4_DIR / "hog_features.npy",
    hog_features
)

print("HOG outputs saved successfully.")

In [ ]:
## Experiment 4.2: SIFT Keypoint Detection

### Objective

To detect distinctive keypoints and calculate feature
descriptors using the Scale-Invariant Feature Transform
(SIFT) algorithm.

SIFT identifies important image regions that can be used
for image matching and object recognition.

In [ ]:
# Create SIFT detector
sift = cv2.SIFT_create()

# Convert photographs to grayscale
gray1 = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY)
gray2 = cv2.cvtColor(second, cv2.COLOR_RGB2GRAY)

# Detect keypoints and descriptors
keypoints1, descriptors1 = sift.detectAndCompute(
    gray1, None
)

keypoints2, descriptors2 = sift.detectAndCompute(
    gray2, None
)

print("SIFT detection completed!")
print("Keypoints in Image 1:", len(keypoints1))
print("Keypoints in Image 2:", len(keypoints2))

In [ ]:
sift_image1 = cv2.drawKeypoints(
    original,
    keypoints1,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

sift_image2 = cv2.drawKeypoints(
    second,
    keypoints2,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(sift_image1)
plt.title("SIFT Keypoints - Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(sift_image2)
plt.title("SIFT Keypoints - Second Image")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
## Experiment 4.3: SIFT Feature Matching

### Objective

To match SIFT descriptors between two photographs
using the Brute Force Matcher.

Lowe's Ratio Test is used to filter unreliable matches
and retain distinctive feature correspondences.

In [ ]:
# Check descriptors
if descriptors1 is None or descriptors2 is None:
    raise ValueError("SIFT descriptors could not be extracted.")

# Brute Force Matcher
bf = cv2.BFMatcher(cv2.NORM_L2)

# K-nearest neighbor matching
matches = bf.knnMatch(
    descriptors1,
    descriptors2,
    k=2
)

# Lowe's Ratio Test
good_matches = []

for pair in matches:
    if len(pair) == 2:
        m, n = pair
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

print("Total matches:", len(matches))
print("Good matches:", len(good_matches))


In [ ]:
matched_image = cv2.drawMatches(
    original,
    keypoints1,
    second,
    keypoints2,
    good_matches,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(16, 8))
plt.imshow(matched_image)
plt.title("SIFT Feature Matching")
plt.axis("off")
plt.show()

In [ ]:
## Experiment 4.4: Rotation and Resizing Experiment

### Objective

To evaluate the performance of SIFT feature matching
after applying rotation and resizing transformations.

The experiment demonstrates the scale and rotation
invariance properties of SIFT.

In [ ]:
height, width = second.shape[:2]

# Rotate image by 30 degrees
rotation_matrix = cv2.getRotationMatrix2D(
    (width // 2, height // 2),
    30,
    1.0
)

rotated_image = cv2.warpAffine(
    second,
    rotation_matrix,
    (width, height)
)

# Resize image to 70%
resized_image = cv2.resize(
    second,
    None,
    fx=0.7,
    fy=0.7
)

plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
plt.imshow(second)
plt.title("Original Second Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(rotated_image)
plt.title("Rotated Image (30°)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(resized_image)
plt.title("Resized Image (70%)")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def sift_match_count(reference, transformed):

    ref_gray = cv2.cvtColor(
        reference, cv2.COLOR_RGB2GRAY
    )

    trans_gray = cv2.cvtColor(
        transformed, cv2.COLOR_RGB2GRAY
    )

    kp1, des1 = sift.detectAndCompute(ref_gray, None)
    kp2, des2 = sift.detectAndCompute(trans_gray, None)

    if des1 is None or des2 is None:
        return len(kp1), len(kp2), 0

    matches = bf.knnMatch(des1, des2, k=2)

    good = [
        m for m, n in matches
        if m.distance < 0.75 * n.distance
    ]

    return len(kp1), len(kp2), len(good)


# Compare original, rotated and resized images
experiments_sift = {
    "Original Image": second,
    "Rotated Image": rotated_image,
    "Resized Image": resized_image
}

sift_results = []

for name, image in experiments_sift.items():

    kp_ref, kp_test, good = sift_match_count(
        original, image
    )

    sift_results.append({
        "Transformation": name,
        "Keypoints in Reference": kp_ref,
        "Keypoints in Test": kp_test,
        "Good Matches": good
    })

sift_df = pd.DataFrame(sift_results)

display(sift_df)

In [ ]:
# Save HOG and SIFT images
cv2.imwrite(
    str(UNIT4_DIR / "sift_keypoints_original.png"),
    cv2.cvtColor(sift_image1, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    str(UNIT4_DIR / "sift_keypoints_second.png"),
    cv2.cvtColor(sift_image2, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    str(UNIT4_DIR / "sift_matching.png"),
    cv2.cvtColor(matched_image, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    str(UNIT4_DIR / "rotated_image.png"),
    cv2.cvtColor(rotated_image, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    str(UNIT4_DIR / "resized_image.png"),
    cv2.cvtColor(resized_image, cv2.COLOR_RGB2BGR)
)

sift_df.to_csv(
    UNIT4_DIR / "sift_comparison.csv",
    index=False
)

print("Unit 4 outputs saved successfully!")

In [ ]:
## Unit 4: Conclusion

HOG feature extraction was successfully performed to
represent image gradients and edge orientations.

SIFT was used to detect keypoints and generate descriptors
from both photographs.

Brute Force Matching and Lowe's Ratio Test were used to
identify corresponding features.

Additional experiments were conducted using rotation
and resizing transformations.

The results demonstrate how feature extraction and
matching techniques help identify similarities between
images under different transformations.

In [ ]:
# UNIT 5: OBJECT RECOGNITION AND MOTION ESTIMATION

## Experiment 5.1: Viola-Jones Face Detection

### Objective

To detect human faces using the pretrained Haar Cascade
classifier based on the Viola-Jones object detection framework.

The algorithm identifies facial regions and displays
rectangular bounding boxes around detected faces.

In [ ]:
import cv2
import os

cascade_path = os.path.join(
    cv2.data.haarcascades,
    "haarcascade_frontalface_default.xml"
)

print("OpenCV version:", cv2.__version__)
print("Cascade directory:", cv2.data.haarcascades)
print("File exists:", os.path.exists(cascade_path))
print("File path:", cascade_path)

In [ ]:
import urllib.request
from pathlib import Path

# Create a local models directory
MODEL_DIR = Path.cwd() / "models"
MODEL_DIR.mkdir(exist_ok=True)

cascade_file = MODEL_DIR / "haarcascade_frontalface_default.xml"

# Download if not already available
if not cascade_file.exists():

    url = (
        "https://raw.githubusercontent.com/"
        "opencv/opencv/master/data/haarcascades/"
        "haarcascade_frontalface_default.xml"
    )

    urllib.request.urlretrieve(url, cascade_file)

print("Cascade file downloaded:", cascade_file.exists())

In [ ]:
face_cascade = cv2.CascadeClassifier(str(cascade_file))

if face_cascade.empty():
    raise RuntimeError("Haar Cascade could not be loaded.")

print("Haar Cascade loaded successfully!")

In [ ]:
face_gray = cv2.cvtColor(
    original,
    cv2.COLOR_RGB2GRAY
)

faces = face_cascade.detectMultiScale(
    face_gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(30, 30)
)

haar_result = original.copy()

for (x, y, w, h) in faces:

    cv2.rectangle(
        haar_result,
        (x, y),
        (x + w, y + h),
        (0, 255, 0),
        3
    )

    cv2.putText(
        haar_result,
        "Face",
        (x, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

print("Number of faces detected:", len(faces))

plt.figure(figsize=(8, 7))
plt.imshow(haar_result)
plt.title("Viola-Jones Face Detection")
plt.axis("off")
plt.show()

In [ ]:
from pathlib import Path

desk_path = IMAGE_DIR / "desk_image.jpg"

print("Image exists:", desk_path.exists())
print("Image path:", desk_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

# Load desk photograph
desk_img = cv2.imread(str(desk_path))

if desk_img is None:
    raise FileNotFoundError("Cannot load desk_image.jpg")

desk_image = cv2.cvtColor(desk_img, cv2.COLOR_BGR2RGB)

# Load YOLO model
model = YOLO("yolo11n.pt")

# Perform object detection
results = model.predict(
    source=desk_image,
    conf=0.20,
    verbose=False
)

# Generate annotated image
annotated = results[0].plot()
annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

# Display output
plt.figure(figsize=(14, 9))
plt.imshow(annotated)
plt.title("YOLO Object Detection - Desk Image")
plt.axis("off")
plt.show()

# Extract detection information
detections = []

for box in results[0].boxes:
    class_id = int(box.cls[0])

    detections.append({
        "Object": model.names[class_id],
        "Confidence": round(float(box.conf[0]), 3)
    })

# Display table
detection_df = pd.DataFrame(detections)

display(detection_df)

print("Total objects detected:", len(detections))

# Save output
UNIT5_DIR.mkdir(parents=True, exist_ok=True)

cv2.imwrite(
    str(UNIT5_DIR / "desk_yolo_detection.png"),
    cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
)

detection_df.to_csv(
    UNIT5_DIR / "desk_detection_results.csv",
    index=False
)

print("Detection results saved successfully!")

In [ ]:
%pip install ultralytics

In [ ]:
%pip install ultralytics --no-deps


In [ ]:
import cv2
import torch
import ultralytics

print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)

print("All imports successful!")

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Project directories
DATASET_DIR = IMAGE_DIR
OUTPUT_DIR = OUTPUT_ROOT
UNIT5_DIR = OUTPUT_DIR / "unit_5_object_detection"
UNIT5_DIR.mkdir(parents=True, exist_ok=True)

# Load desk photograph
desk_path = DATASET_DIR / "desk_image.jpg"

desk_img = cv2.imread(str(desk_path))

if desk_img is None:
    raise FileNotFoundError(f"Image not found: {desk_path}")

desk_image = cv2.cvtColor(desk_img, cv2.COLOR_BGR2RGB)

# Load YOLO model
model = YOLO("yolo11n.pt")

# Detect objects
desk_results = model.predict(
    source=desk_image,
    conf=0.20,
    iou=0.45,
    verbose=False
)

# Annotate image
desk_annotated = desk_results[0].plot()
desk_annotated = cv2.cvtColor(
    desk_annotated, cv2.COLOR_BGR2RGB
)

# Display result
plt.figure(figsize=(14, 9))
plt.imshow(desk_annotated)
plt.title("YOLO Object Detection - Desk Image")
plt.axis("off")
plt.show()

# Extract detection details
detections = []

for box in desk_results[0].boxes:
    class_id = int(box.cls[0])

    detections.append({
        "Object": model.names[class_id],
        "Confidence": round(float(box.conf[0]), 3)
    })

desk_detection_df = pd.DataFrame(
    detections,
    columns=["Object", "Confidence"]
)

display(desk_detection_df)

print("Total detected objects:", len(detections))

# Save output
cv2.imwrite(
    str(UNIT5_DIR / "desk_yolo_detection.png"),
    cv2.cvtColor(desk_annotated, cv2.COLOR_RGB2BGR)
)

desk_detection_df.to_csv(
    UNIT5_DIR / "desk_detection_results.csv",
    index=False
)

print("Desk image YOLO experiment completed and saved!")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CPU available:", torch.backends.cpu.get_cpu_capability())

torch.set_num_threads(1)

print("PyTorch initialized successfully!")

In [ ]:
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("Import successful!")

In [ ]:
import cv2
import torch
from ultralytics import YOLO

# Limit PyTorch CPU threads
torch.set_num_threads(1)

# Load lightweight YOLO model
model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully!")

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

desk_path = IMAGE_DIR / "desk_image.jpg"

desk_img = cv2.imread(str(desk_path))

assert desk_img is not None, "Desk image not found!"

# Resize image to reduce processing requirements
height, width = desk_img.shape[:2]

scale = min(640 / width, 640 / height, 1)

desk_small = cv2.resize(
    desk_img,
    (int(width * scale), int(height * scale))
)

# Run inference on CPU
results = model.predict(
    source=desk_small,
    imgsz=640,
    conf=0.25,
    device="cpu",
    verbose=False
)

print("YOLO inference completed!")

In [ ]:
annotated = results[0].plot()

annotated_rgb = cv2.cvtColor(
    annotated, cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(12, 8))
plt.imshow(annotated_rgb)
plt.title("YOLO Object Detection - Desk Image")
plt.axis("off")
plt.show()

detections = []

for box in results[0].boxes:
    detections.append({
        "Object": model.names[int(box.cls[0])],
        "Confidence": round(float(box.conf[0]), 3)
    })

detection_df = pd.DataFrame(
    detections,
    columns=["Object", "Confidence"]
)

display(detection_df)

UNIT5_DIR.mkdir(parents=True, exist_ok=True)

cv2.imwrite(
    str(UNIT5_DIR / "desk_yolo_detection.png"),
    annotated
)

detection_df.to_csv(
    UNIT5_DIR / "desk_detection_results.csv",
    index=False
)

print("YOLO results saved!")